In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh



In [ ]:
import subprocess
import threading
import time

import requests


def wait_for_ollama(timeout=60, interval=1):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
            if response.ok:
                return True
        except requests.RequestException:
            time.sleep(interval)
    return False


def launch_ollama():
    # Start Ollama as a background process and wait until it is reachable.
    def run():
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

    threading.Thread(target=run, daemon=True).start()
    if not wait_for_ollama():
        raise RuntimeError("Ollama did not become ready on http://127.0.0.1:11434")
    print("Ollama is serving in the background!")


launch_ollama()

In [ ]:
!ollama pull bge-m3

In [ ]:
!pip install requests
!pip install pandas

In [ ]:
import requests

In [ ]:
import requests


def create_embeddings(text_list):
    # embeddings = []
    # for idx, text in enumerate(text_list):
    #     try:
    #         res = requests.post(
    #             "http://localhost:11434/api/embed",
    #             json={"model": "bge-m3", "input": [text]},
    #         )
    #         if res.status_code != 200:
    #             print(f"❌ Chunk at index {idx} failed! Content: {text[:100]}...")
    #             embeddings.append(0)
    #         else:
    #             embedding = res.json()["embeddings"]
    #             embeddings.append(embedding)
    #     except Exception as e:
    #         print(f"Error on index {idx}: {e}")

    # return embeddings
    response = requests.post("http://localhost:11434/api/embed", json={
        "model": "bge-m3",
        "input": text_list
    })
    
    embeddings = response.json()["embeddings"]
    return embeddings

In [ ]:
# response = create_embeddings(["What is RAG-based AI?", "How does it work?"])
# print(response)

import os
import json
import pandas as pd
file_name = "merged_jsons" if os.path.exists("merged_jsons") else "jsons"
jsons = os.listdir(file_name)
my_dict = []

    
    
for json_file in jsons:
    with open(os.path.join(file_name, json_file), "r") as f:
        content = json.load(f)
    text_list = [c["text"] for c in content["chunks"]] 
    embeddings = create_embeddings(text_list)

    for i,chunk in enumerate(content["chunks"]):
        chunk['embedding'] = embeddings[i]
        my_dict.append(chunk)


df = pd.DataFrame.from_records(my_dict)
print(df)


In [ ]:
import joblib
joblib.dump(df, "embeddings.joblib")

In [ ]:
import joblib
df_loaded = joblib.load("embeddings.joblib")
print(df_loaded)

In [ ]:
incoming_query = "where is api used"
question_embedding = create_embeddings([incoming_query])[0]
print(question_embedding)

In [ ]:
import numpy as np

In [ ]:
print(np.vstack(df["embedding"].values))
print(np.vstack(df["embedding"].shape))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(
    np.vstack(df_loaded["embedding"].values),
    [question_embedding]
).flatten()
top_results = 5
result_indx = similarities.argsort()[-top_results:][::-1]

new_df = df_loaded.loc[result_indx]
print(new_df)

In [ ]:
prompt = f'''Here are video subtitle chunks containing video title, video number, start time in seconds, end time in seconds, the text at that time:

{new_df[["title", "number", "start", "end", "text"]].to_json(orient="records")}
---------------------------------
"{incoming_query}"
User asked this question related to the video chunks, you have to answer in a human way (by analyzing all chunks(like where it is mostly mentioned))and tell which video might contain the needed information (don't mention chunk number) by giving the title(you can see that many chunks have same title which means they are part of same video) and the timestamp(may merge with other and convert to minutes). (this output will be directly shared to the user so don't say any words describing the data(like chunks or anything) just mention the title timestamp and the reasoning)
'''

print(prompt)

In [ ]:
!ollama pull llama3.2

In [ ]:
def inference(prompt):
    r = requests.post("http://localhost:11434/api/generate", json={
        # "model": "deepseek-r1",
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False
    })
    
    return r.json()["response"]

In [ ]:
print(inference(prompt))

In [ ]:
for idx, row in new_df.iterrows():
    print(f"Chunk: {row['text'][:100]}... | Similarity: {similarities[idx]:.4f}")

In [ ]:
import os

def stop_ollama():
    # Sends a termination signal to any running ollama process
    os.system("pkill ollama")
    print(" Ollama server has been stopped.")

# stop_ollama()